# HERMES on AutoDL — 50-video StreamingBench reproduction

**Default one-pass scope:** official HERMES at KV budgets **4,000 / 6,000 / 50,000**, samples **201–250** (50 videos, 250 questions), paired analysis, video-cluster bootstrap, figures, and a checked evidence archive.

The 50K condition is a **near-uncompressed HERMES control**, not native LLaVA. The incomplete native full-prefix baseline and optional 498-video benchmark are intentionally excluded: neither is required for this reproduction and both prevent a reliable one-pass run.

### AutoDL setup
1. Use a CUDA GPU with at least 16 GB VRAM (24 GB recommended) and enough `/root/autodl-tmp` space (35 GB free minimum; more is safer).
2. Upload this notebook and open it in JupyterLab. Use the normal Python 3 kernel for the notebook; inference uses its own pinned Python 3.12 environment.
3. Select **Run → Run All Cells** once. Do not start multiple copies concurrently.
4. Downloads, environment, videos, logs and outputs persist under `/root/autodl-tmp/hermes_autodl`. A rerun reuses only fully validated CSVs.

Actual runtime depends on the GPU and network. A clean run takes roughly a few hours, not a few minutes. Any failed integrity, dependency, annotation or result check stops the notebook rather than producing misleading output.

## 1. Configuration and AutoDL preflight

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, re, zipfile, hashlib, shutil, time, copy, platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy.stats import binomtest
import matplotlib.pyplot as plt
from IPython.display import display

MODEL_ID = 'llava-hf/llava-onevision-qwen2-0.5b-ov-hf'
HERMES_URL = 'https://github.com/haowei-freesky/HERMES.git'
HERMES_SHA = '8d699b16a6bedb9086c1b39ec4253c6a1d1ce789'
TF_SHA = '66bc4def9505fa7c7fe4aa7a248c34a026bb552b'
DATASET_ID = 'mjuicem/StreamingBench'
FPS, SEED = 0.5, 2024
KV_BUDGETS = (4000, 6000, 50000)
BOOTSTRAP_REPS = 10000
MIN_FREE_GIB, MIN_VRAM_GIB = 35, 15

WORK = Path('/root/autodl-tmp/hermes_autodl')
ROOT = WORK / 'HERMES'
RESULTS, DATA = ROOT / 'results', ROOT / 'data'
EXPORTS, FIGURES = WORK / 'exports', WORK / 'exports' / 'figures'
LOGS = WORK / 'logs'
VENV = WORK / 'venvs' / 'hermes-py312'
PY312 = VENV / 'bin' / 'python'
ARCHIVE_CACHE = WORK / 'downloads'
VIDEO_DIR = WORK / 'videos' / 'streamingbench_201_250'
HF_HOME = WORK / 'cache' / 'huggingface'
MODEL_CACHE = HF_HOME / 'hub'
UV_CACHE = WORK / 'cache' / 'uv'
for p in (WORK, RESULTS, DATA, EXPORTS, FIGURES, LOGS, ARCHIVE_CACHE, VIDEO_DIR, HF_HOME, MODEL_CACHE, UV_CACHE):
    p.mkdir(parents=True, exist_ok=True)

def run(args, *, cwd=None, env=None, check=True):
    proc = subprocess.run([str(a) for a in args], cwd=cwd, env=env,
                          text=True, capture_output=True)
    if check and proc.returncode:
        raise RuntimeError('Command failed: ' + repr(args) + '\n' + (proc.stdout + proc.stderr)[-6000:])
    return proc

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def result_path(kv):
    return RESULTS / f'streamingbench_50_kv{kv}' / '1_0.csv'

assert Path('/root/autodl-tmp').is_dir(), 'AutoDL data disk /root/autodl-tmp is missing.'
free_gib = shutil.disk_usage(WORK).free / 1024**3
assert free_gib >= MIN_FREE_GIB, f'Only {free_gib:.1f} GiB free; free at least {MIN_FREE_GIB} GiB.'
smi = run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=False)
assert smi.returncode == 0 and smi.stdout.strip(), 'No NVIDIA GPU visible: ' + smi.stderr
print('GPU:', smi.stdout.strip())
mem_mib = float(re.search(r'(\d+(?:\.\d+)?)\s*MiB', smi.stdout).group(1))
assert mem_mib / 1024 >= MIN_VRAM_GIB, f'GPU has {mem_mib/1024:.1f} GiB; at least {MIN_VRAM_GIB} GiB required.'
print('Notebook Python:', sys.version.split()[0])
print('Workspace:', WORK, '| free:', round(free_gib, 1), 'GiB')
print('Default scope: 3 HERMES budgets × 250 questions; native/full-benchmark stages disabled.')

## 2. Create the pinned Python 3.12 inference environment

This does not alter AutoDL’s system Python or the Jupyter kernel.

In [ ]:
try:
    run([sys.executable, '-m', 'uv', '--version'])
except RuntimeError:
    run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
UV = [sys.executable, '-m', 'uv']
INSTALL_ENV = {**os.environ, 'UV_CACHE_DIR': str(UV_CACHE), 'UV_LINK_MODE': 'copy', 'TMPDIR': str(WORK)}
if not PY312.exists():
    # uv supplies a managed CPython 3.12 when the image does not include one.
    run(UV + ['venv', str(VENV), '--python', '3.12', '--seed'], env=INSTALL_ENV)
assert '3.12.' in run([PY312, '--version']).stdout

def uv_install(packages, extra_args=()):
    proc = run(UV + ['pip', 'install', '--python', str(PY312)] + list(extra_args) + list(packages), env=INSTALL_ENV)
    if proc.stderr:
        print(proc.stderr[-1200:])

uv_install(['torch==2.10.0+cu128'], ['--index-url', 'https://download.pytorch.org/whl/cu128'])
uv_install(['numpy==1.26.4', 'tokenizers==0.19.1', 'huggingface-hub==0.34.4',
            'accelerate==0.34.2', 'decord==0.6.0', 'logzero', 'pillow', 'sentencepiece',
            'safetensors', 'packaging', 'filelock', 'regex', 'requests', 'pyyaml', 'tqdm',
            'fsspec', 'protobuf', 'einops', 'psutil', 'pandas', 'scipy', 'av'])
uv_install(['git+https://github.com/huggingface/transformers.git@' + TF_SHA], ['--no-deps'])
print('Pinned environment ready:', PY312)

In [ ]:
check_code = r"""
import sys, json, inspect, torch, transformers, tokenizers, huggingface_hub
import importlib.metadata as md
from transformers.models.qwen2.modeling_qwen2 import Qwen2RotaryEmbedding
assert sys.version_info[:2] == (3, 12), sys.version
assert torch.__version__ == '2.10.0+cu128', torch.__version__
assert torch.cuda.is_available(), 'The inference interpreter cannot see CUDA'
assert transformers.__version__ == '4.45.0.dev0', transformers.__version__
assert tokenizers.__version__ == '0.19.1'
assert huggingface_hub.__version__ == '0.34.4'
meta = md.distribution('transformers').read_text('direct_url.json')
assert meta and json.loads(meta)['vcs_info']['commit_id'] == '66bc4def9505fa7c7fe4aa7a248c34a026bb552b'
assert 'seq_len' in str(inspect.signature(Qwen2RotaryEmbedding.forward))
print('Inference Python:', sys.executable)
print('Torch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))
print('Transformers:', transformers.__version__, '| pinned source: PASS')
"""
model_env = {**os.environ, 'HF_HOME': str(HF_HOME), 'HF_HUB_DISABLE_XET': '1',
             'HF_HUB_ENABLE_HF_TRANSFER': '0', 'CUDA_VISIBLE_DEVICES': '0'}
print(run([PY312, '-c', check_code], env=model_env).stdout)

## 3. Obtain and verify the official HERMES source

In [ ]:
if not (ROOT / '.git').exists():
    allowed = {'results', 'data'}
    if ROOT.exists() and any(p.name not in allowed for p in ROOT.iterdir()):
        raise RuntimeError(f'{ROOT} contains unrecognized files without Git metadata; move it aside before continuing.')
    staging = WORK / 'HERMES-source-staging'
    if staging.exists():
        shutil.rmtree(staging)
    run(['git', 'clone', HERMES_URL, str(staging)])
    shutil.copytree(staging, ROOT, dirs_exist_ok=True)
    shutil.rmtree(staging)
now = run(['git', 'rev-parse', 'HEAD'], cwd=ROOT).stdout.strip()
if now != HERMES_SHA:
    run(['git', 'checkout', '--detach', HERMES_SHA], cwd=ROOT)
    now = run(['git', 'rev-parse', 'HEAD'], cwd=ROOT).stdout.strip()
assert now == HERMES_SHA
changed = run(['git', 'diff', '--name-only', 'HEAD', '--'], cwd=ROOT).stdout.splitlines()
unexpected = [p for p in changed if not (p.endswith('.pyc') and '__pycache__/' in p)]
assert not unexpected, f'Unexpected tracked source modifications: {unexpected}'
for rel in ('inference/llavaov_hermes.py', 'inference/reindex_1d.py', 'video_qa/base.py'):
    assert (ROOT / rel).is_file(), rel
    print(rel, sha256(ROOT / rel))
reindex_tail = (ROOT / 'inference/reindex_1d.py').read_text().split('def apply_rotary_delta_to_keys_only')[-1][:230]
assert 'return key_states' not in reindex_tail, 'Disabled RoPE-delta debug patch detected.'
print('HERMES revision:', now, '| source fidelity: PASS')

## 4. Download the model and record its local snapshot identity

In [ ]:
model_downloader = WORK / 'download_model.py'
model_downloader.write_text(r"""
import json, sys
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id=sys.argv[1], cache_dir=sys.argv[2],
    allow_patterns=['*.json','*.safetensors','*.txt','*.model','*.vocab','tokenizer.*','merges.txt'],
    ignore_patterns=['onnx/*','*.onnx','*.bin','*.msgpack'])
print('MODEL_SNAPSHOT_JSON=' + json.dumps(str(path)))
""")
proc = run([PY312, str(model_downloader), MODEL_ID, str(MODEL_CACHE)], env=model_env)
lines = [s for s in proc.stdout.splitlines() if s.startswith('MODEL_SNAPSHOT_JSON=')]
assert lines, 'Model snapshot was not reported: ' + proc.stdout[-1500:]
MODEL_SNAPSHOT = Path(json.loads(lines[-1].split('=', 1)[1]))
assert MODEL_SNAPSHOT.is_dir() and list(MODEL_SNAPSHOT.glob('*.safetensors')), 'Checkpoint weights missing.'
link = ROOT / 'models' / 'llava-onevision-qwen2-0.5b-ov-hf'
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() and link.resolve() != MODEL_SNAPSHOT.resolve():
    link.unlink()
if not link.exists():
    link.symlink_to(MODEL_SNAPSHOT, target_is_directory=True)
assert (link / 'config.json').is_file()
MODEL_DIR = link
print('Model:', MODEL_DIR)
print('Resolved snapshot:', MODEL_SNAPSHOT)
print('Snapshot identity:', MODEL_SNAPSHOT.name)

## 5. Prepare StreamingBench 201–250 and validate question identity

Repeated question text is possible, so every result is keyed by `(video_id, qa_index)` and checked against the source annotation ordinal.

In [ ]:
ANNOTATION_MASTER = ROOT / 'data' / 'streamingbench' / 'streamingbench_realtime.json'
annotations = json.loads(ANNOTATION_MASTER.read_text())
assert isinstance(annotations, list) and len(annotations) == 498
assert sum(len(x['conversations']) for x in annotations) == 2495
SHARD_50 = 'Real-Time Visual Understanding_201-250.zip'

def safe_extract(z, output_dir):
    output_dir = Path(output_dir)
    base = output_dir.resolve()
    total = 0
    for item in z.infolist():
        target = (output_dir / item.filename).resolve()
        assert base == target or base in target.parents, 'ZIP path traversal: ' + item.filename
        assert not item.is_dir() or not item.filename.startswith('/'), 'Unsafe ZIP entry'
        total += item.file_size
    assert total <= 30 * 1024**3, f'Archive expands to an unexpected {total/1024**3:.1f} GiB'
    z.extractall(output_dir)

def fetch_extract(archive_name):
    if any(VIDEO_DIR.rglob('*.mp4')):
        return VIDEO_DIR
    downloader = WORK / 'download_shard.py'
    downloader.write_text(r"""
import sys, json
from huggingface_hub import hf_hub_download
path = hf_hub_download(repo_id=sys.argv[1], repo_type='dataset', filename=sys.argv[2], local_dir=sys.argv[3])
print('ARCHIVE_JSON=' + json.dumps(str(path)))
""")
    proc = run([PY312, str(downloader), DATASET_ID, archive_name, str(ARCHIVE_CACHE)], env=model_env)
    lines = [s for s in proc.stdout.splitlines() if s.startswith('ARCHIVE_JSON=')]
    assert lines, 'Archive path not returned: ' + proc.stdout[-1500:]
    archive = Path(json.loads(lines[-1].split('=', 1)[1]))
    assert zipfile.is_zipfile(archive), archive
    with zipfile.ZipFile(archive) as z:
        safe_extract(z, VIDEO_DIR)
    assert any(VIDEO_DIR.rglob('*.mp4')), 'Extracted archive contains no MP4 videos.'
    archive.unlink(missing_ok=True)
    return VIDEO_DIR

def annotation_for_directory(path):
    candidates = {}
    for p in Path(path).rglob('*'):
        if p.is_file() and p.suffix.lower() in {'.mp4','.avi','.mov','.mkv'} and not p.name.startswith('._'):
            matches = re.findall(r'sample_(\d+)', str(p))
            if matches:
                sample_id = int(matches[-1])
                assert sample_id not in candidates, f'Multiple videos for sample_{sample_id}'
                candidates[sample_id] = p
    mapped = []
    for item in annotations:
        match = re.search(r'sample_(\d+)_real\.mp4', str(item['video_path']))
        if match and int(match.group(1)) in candidates:
            record = copy.deepcopy(item)
            record['video_path'] = str(candidates[int(match.group(1))])
            mapped.append((int(match.group(1)), record))
    return [r for _, r in sorted(mapped, key=lambda x: x[0])]

shard50_dir = fetch_extract(SHARD_50)
matched50 = annotation_for_directory(shard50_dir)
assert len(matched50) == 50, f'Expected 50 matched videos, got {len(matched50)}'
assert sum(len(x['conversations']) for x in matched50) == 250
assert all(Path(x['video_path']).is_file() for x in matched50)
ANNOTATION_50 = DATA / 'streamingbench_50_autodl.json'
ANNOTATION_50.write_text(json.dumps(matched50, indent=2))
ANNOTATION_LOOKUP = {str(item['video_id']): item for item in matched50}
print('Prepared:', ANNOTATION_50, '| videos: 50 | questions: 250')

In [ ]:
import ast
KEY = ['video_id', 'qa_index']

def validate_annotation_positions(d, lookup, label='CSV'):
    tested = 0
    for _, row in d.iterrows():
        vid = str(row['video_id'])
        assert vid in lookup, (label, vid, 'unexpected video')
        position = int(row['qa_index'])
        qa_list = lookup[vid]['conversations']
        assert 0 <= position < len(qa_list), (label, vid, position, 'out of bounds')
        qa = qa_list[position]
        assert str(row['question']).strip() == str(qa['question']).strip(), (label, vid, position, 'question-order mismatch')
        if 'choices' in d:
            raw = row['choices']
            csv_choices = ast.literal_eval(raw) if isinstance(raw, str) else raw
            assert isinstance(csv_choices, (tuple, list))
            assert [str(c).strip() for c in csv_choices] == [str(c).strip() for c in qa['choices']], (label, vid, position, 'choices differ')
        hits = [j for j, answer in enumerate(qa['choices']) if str(answer).strip() == str(qa['answer']).strip()]
        assert len(hits) == 1, (label, vid, position, 'ambiguous gold answer')
        if 'correct_choice' in d:
            assert str(row['correct_choice']).strip().upper() == 'ABCD'[hits[0]], (label, vid, position, 'gold-letter mismatch')
        tested += 1
    print(f'Annotation audit: {label}: {tested}/{len(d)} rows matched to source')

def read_hermes_csv(path, expected_videos=50, expected_rows=250):
    path = Path(path)
    d = pd.read_csv(path).copy()
    assert {'video_id','question','qa_acc'}.issubset(d.columns), list(d.columns)
    d['video_id'] = d['video_id'].astype(str)
    d['question'] = d['question'].astype(str).str.strip()
    d['qa_index'] = d.groupby('video_id', sort=False).cumcount().astype(int)
    assert not d.duplicated(KEY).any(), f'Duplicate positional keys in {path}'
    assert d.video_id.nunique() == expected_videos, (path, d.video_id.nunique())
    assert len(d) == expected_rows, (path, len(d))
    assert d.groupby('video_id').size().eq(5).all(), (path, 'expected five questions per video')
    assert not d[['video_id','question','qa_acc']].isna().any().any()
    validate_annotation_positions(d, ANNOTATION_LOOKUP, str(path))
    values = pd.to_numeric(d.qa_acc, errors='coerce')
    assert values.isin([0,100]).all(), f'Invalid qa_acc in {path}'
    d['correct'] = values.eq(100).astype(int)
    if {'pred_choice','correct_choice'}.issubset(d):
        parsed_ok = d.pred_choice.astype(str).str.strip().str.upper().eq(d.correct_choice.astype(str).str.strip().str.upper())
        print(path.parent.name, 'parsed-choice/qa_acc mismatches:', int(parsed_ok.ne(d.correct.astype(bool)).sum()))
    return d

print('CSV and annotation validators ready.')

## 6. Run or resume the three HERMES configurations

An existing CSV is reused only after all 250 rows pass structural and source-annotation validation. A malformed or partial CSV stops the run for manual preservation; it is never silently treated as complete.

In [ ]:
SUBENV = {**model_env, 'PYTHONPATH': str(ROOT) + os.pathsep + os.environ.get('PYTHONPATH',''),
          'TOKENIZERS_PARALLELISM': 'false', 'PYTORCH_ALLOC_CONF': 'expandable_segments:True'}

def run_hermes(budget):
    output_name = f'streamingbench_50_kv{budget}'
    result_dir = RESULTS / output_name
    csv_path, log_path = result_dir / '1_0.csv', LOGS / (output_name + '.log')
    if csv_path.exists():
        print('Reusing fully validated:', csv_path)
        return read_hermes_csv(csv_path), 0.0, True
    result_dir.mkdir(parents=True, exist_ok=True)
    command = [PY312, '-m', 'video_qa.hermes_vqa', '--model', 'llava_ov_0.5b',
               '--sample_fps', str(FPS), '--save_dir', str(result_dir), '--anno_path', str(ANNOTATION_50),
               '--debug', 'false', '--num_chunks', '1', '--chunk_idx', '0',
               '--kv_size', str(budget), '--streaming', 'true']
    started = time.time()
    print(f'Starting KV={budget}; live output is written to {log_path}', flush=True)
    with log_path.open('w') as handle:
        proc = subprocess.run([str(x) for x in command], cwd=ROOT, env=SUBENV,
                              stdout=handle, stderr=subprocess.STDOUT)
    elapsed_min = (time.time() - started) / 60
    print(output_name, 'return:', proc.returncode, '| elapsed min:', round(elapsed_min, 2), flush=True)
    if proc.returncode:
        print(log_path.read_text(errors='replace')[-5000:])
        raise RuntimeError(f'HERMES failed for KV={budget}; inspect {log_path}')
    return read_hermes_csv(csv_path), elapsed_min, False

run_records, results50 = [], {}
for kv in KV_BUDGETS:
    d, elapsed, reused = run_hermes(kv)
    results50[kv] = d
    run_records.append({'kv_budget': kv, 'questions': len(d), 'videos': d.video_id.nunique(),
                        'accuracy_pct': 100*d.correct.mean(), 'elapsed_min_this_session': elapsed,
                        'reused_validated_csv': reused, 'log': str(LOGS / f'streamingbench_50_kv{kv}.log')})
    print(f'KV={kv}: {d.correct.sum()}/{len(d)} correct ({100*d.correct.mean():.2f}%)', flush=True)
run_summary = pd.DataFrame(run_records)
run_summary.to_csv(EXPORTS / 'run_summary.csv', index=False)
display(run_summary)

## 7. Paired correctness analysis and video-cluster bootstrap

Exact discordance tests are exploratory. Confidence intervals resample whole videos, preserving within-video dependence.

In [ ]:
def paired(a, b, name_a, name_b, reps=BOOTSTRAP_REPS, seed=SEED):
    assert set(map(tuple, a[KEY].values)) == set(map(tuple, b[KEY].values)), 'Question keys differ.'
    aa, bb = a.set_index(KEY).sort_index(), b.set_index(KEY).sort_index()
    for field in ('question','choices','correct_choice','answer','task'):
        if field in aa and field in bb:
            bad = aa[field].fillna('').astype(str).str.strip().ne(bb[field].fillna('').astype(str).str.strip())
            assert not bad.any(), f'Metadata mismatch in {field}: {bad[bad].index.tolist()[:5]}'
    x = a[KEY+['correct']].rename(columns={'correct':'a'}).merge(
        b[KEY+['correct']].rename(columns={'correct':'b'}), on=KEY, validate='one_to_one')
    n11 = int(((x.a==1)&(x.b==1)).sum()); n00 = int(((x.a==0)&(x.b==0)).sum())
    n10 = int(((x.a==1)&(x.b==0)).sum()); n01 = int(((x.a==0)&(x.b==1)).sum())
    exact = binomtest(n10, n10+n01, .5).pvalue if n10+n01 else 1.0
    by_video = x.assign(delta=x.a-x.b).groupby('video_id').delta.agg(['sum','count'])
    sums, counts = by_video['sum'].to_numpy(float), by_video['count'].to_numpy(float)
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, len(sums), size=(reps, len(sums)))
    boot = 100 * sums[draws].sum(axis=1) / counts[draws].sum(axis=1)
    result = {'A':name_a, 'B':name_b, 'n':len(x), 'n_videos':x.video_id.nunique(),
              'both_correct':n11, 'both_wrong':n00, 'A_only_correct':n10, 'B_only_correct':n01,
              'accuracy_A_pct':100*x.a.mean(), 'accuracy_B_pct':100*x.b.mean(),
              'difference_A_minus_B_pp':100*(x.a-x.b).mean(),
              'video_cluster_CI95_low_pp':float(np.quantile(boot,.025)),
              'video_cluster_CI95_high_pp':float(np.quantile(boot,.975)),
              'exact_discordance_p_exploratory':exact}
    x['comparison'] = f'{name_a}_vs_{name_b}'
    x['case_type'] = np.select([x.a.eq(1)&x.b.eq(1), x.a.eq(0)&x.b.eq(0),
                                x.a.eq(1)&x.b.eq(0), x.a.eq(0)&x.b.eq(1)],
                               ['both_correct','both_wrong','A_only_correct','B_only_correct'], default='invalid')
    return result, x

comparison_rows, cases = [], []
for a, b in ((4000,6000),(4000,50000),(6000,50000)):
    row, case = paired(results50[a], results50[b], f'HERMES_{a}', f'HERMES_{b}')
    comparison_rows.append(row); cases.append(case)
paired50 = pd.DataFrame(comparison_rows)
casebook50 = pd.concat(cases, ignore_index=True)
meta_cols = [c for c in KEY+['question','answer','choices','task','pred_answer','pred_choice','correct_choice'] if c in results50[4000]]
casebook50 = casebook50.merge(results50[4000][meta_cols], on=KEY, how='left', validate='many_to_one')
paired50.to_csv(EXPORTS / 'paired_50_tests.csv', index=False)
casebook50.to_csv(EXPORTS / 'paired_50_casebook.csv', index=False)
accuracy50 = pd.DataFrame([{'method':f'HERMES KV={k}', 'kv_budget':k, 'n':len(d),
                            'correct':int(d.correct.sum()), 'accuracy_pct':100*d.correct.mean()}
                           for k,d in results50.items()])
accuracy50.to_csv(EXPORTS / 'hermes_50_accuracy.csv', index=False)
all_three = results50[4000][KEY+['question','correct']].rename(columns={'correct':'ok4'})
for k, col in ((6000,'ok6'),(50000,'ok50')):
    all_three = all_three.merge(results50[k][KEY+['correct']].rename(columns={'correct':col}), on=KEY, validate='one_to_one')
all_three['pattern'] = all_three[['ok4','ok6','ok50']].astype(str).agg(''.join, axis=1)
all_three.to_csv(EXPORTS / 'all_three_50_correctness.csv', index=False)
display(accuracy50.round(3)); display(paired50.round(4))
display(all_three.pattern.value_counts().rename_axis('pattern_4k_6k_50k').to_frame('questions'))

## 8. Task summaries and figures

In [ ]:
task_frames = []
for kv, d in results50.items():
    if 'task' in d:
        t = d.groupby('task').correct.agg(['count','sum','mean']).reset_index()
        t['kv_budget'] = kv; t['accuracy_pct'] = 100*t['mean']
        task_frames.append(t)
if task_frames:
    task50 = pd.concat(task_frames, ignore_index=True)
    task50.to_csv(EXPORTS / 'hermes_50_by_task.csv', index=False)
    display(task50[['task','kv_budget','count','sum','accuracy_pct']].round(3))
else:
    print('No task column in HERMES CSVs; task summary skipped without imputation.')

plt.figure(figsize=(7,4.5))
plt.bar(accuracy50.kv_budget.astype(str), accuracy50.accuracy_pct, color=['#386cb0','#7fc97f','#beaed4'])
plt.ylim(0,100); plt.xlabel('HERMES KV budget'); plt.ylabel('Accuracy (%)')
plt.title('StreamingBench samples 201–250 (n=250 questions)')
for x, y in zip(accuracy50.kv_budget.astype(str), accuracy50.accuracy_pct):
    plt.text(x, y+1, f'{y:.1f}%', ha='center')
plt.tight_layout(); plt.savefig(FIGURES/'accuracy_50.png', dpi=180); plt.show()

plt.figure(figsize=(7,4.5))
labels = ['4K − 6K', '4K − 50K', '6K − 50K']
y = paired50.difference_A_minus_B_pp.to_numpy()
lo = y-paired50.video_cluster_CI95_low_pp.to_numpy(); hi = paired50.video_cluster_CI95_high_pp.to_numpy()-y
plt.errorbar(labels, y, yerr=[lo,hi], fmt='o', capsize=5, color='#386cb0')
plt.axhline(0,color='black',lw=.8); plt.ylabel('Accuracy difference (percentage points)')
plt.title('Paired differences with 95% video-cluster bootstrap CI')
plt.tight_layout(); plt.savefig(FIGURES/'paired_differences_50.png', dpi=180); plt.show()

## 9. Final integrity checks and evidence archive

In [ ]:
# Re-read every deliverable from disk; no result is inferred from a missing file.
for kv in KV_BUDGETS:
    checked = read_hermes_csv(result_path(kv), 50, 250)
    assert len(checked) == 250 and checked.video_id.nunique() == 50
required = [EXPORTS/'hermes_50_accuracy.csv', EXPORTS/'paired_50_tests.csv',
            EXPORTS/'paired_50_casebook.csv', EXPORTS/'all_three_50_correctness.csv',
            EXPORTS/'run_summary.csv', FIGURES/'accuracy_50.png', FIGURES/'paired_differences_50.png']
assert all(p.is_file() and p.stat().st_size > 0 for p in required)
assert len(pd.read_csv(EXPORTS/'paired_50_tests.csv')) == 3

provenance = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'scope': 'StreamingBench samples 201-250; HERMES KV 4000/6000/50000',
    'videos': 50, 'questions_per_budget': 250, 'sample_fps': FPS, 'seed': SEED,
    'model_id': MODEL_ID, 'model_snapshot_identity': MODEL_SNAPSHOT.name,
    'hermes_url': HERMES_URL, 'hermes_revision': HERMES_SHA, 'transformers_revision': TF_SHA,
    'native_baseline': 'not run; incomplete historical baseline deliberately excluded',
    'full_498_video_benchmark': 'not run',
    'statistical_note': 'Exact discordance tests exploratory; CIs use video-cluster bootstrap.'
}
(EXPORTS/'provenance.json').write_text(json.dumps(provenance, indent=2))

payload = [ANNOTATION_50, EXPORTS/'provenance.json'] + required
payload += [result_path(k) for k in KV_BUDGETS]
payload += [LOGS/f'streamingbench_50_kv{k}.log' for k in KV_BUDGETS if (LOGS/f'streamingbench_50_kv{k}.log').exists()]
if (EXPORTS/'hermes_50_by_task.csv').exists():
    payload.append(EXPORTS/'hermes_50_by_task.csv')
payload = sorted(set(Path(p).resolve() for p in payload))
manifest_rows = []
for p in payload:
    assert p.is_file() and p.stat().st_size > 0, p
    manifest_rows.append({'path': str(p.relative_to(WORK)), 'bytes':p.stat().st_size, 'sha256':sha256(p)})
manifest = pd.DataFrame(manifest_rows)
manifest_path = EXPORTS/'manifest_sha256.csv'
manifest.to_csv(manifest_path, index=False)
payload.append(manifest_path.resolve())
archive = WORK/'HERMES_AutoDL_50_evidence.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in sorted(set(payload)):
        z.write(p, arcname=str(p.relative_to(WORK)))
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None, 'ZIP CRC check failed'
    assert 'exports/manifest_sha256.csv' in z.namelist()
archive_hash = sha256(archive)
(EXPORTS/'archive_sha256.txt').write_text(f'{archive_hash}  {archive.name}\n')
print('FINAL STATUS: PASS')
print('Evidence archive:', archive)
print('Archive bytes:', archive.stat().st_size, '| SHA-256:', archive_hash)
print('All three raw result CSVs, analysis tables, figures, available logs and provenance are included.')
print('Optional backup: copy the ZIP to /root/autodl-fs after this cell completes.')